# RL Autopilot Training

This notebook demonstrates the AEGIS RL autopilot for bias mitigation.
A PPO (Proximal Policy Optimization) agent learns to adjust model
thresholds and feature weights to improve fairness metrics.

**Goals:**
1. Load dataset and train a baseline model
2. Compute initial fairness metrics
3. Set up PPO environment and agent
4. Train the PPO agent
5. Compare before/after fairness metrics
6. Visualize training curves
7. Summary

In [ ]:
# Cell 1: Imports
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
import time

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

# Project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    HAS_TORCH = True
except ImportError:
    print("PyTorch: not installed (PPO training will be unavailable)")
    HAS_TORCH = False

In [ ]:
# Cell 2: Load dataset and train baseline model
n_samples = 1000
n_features = 10

X_raw, y = make_classification(
    n_samples=n_samples,
    n_features=n_features,
    n_informative=7,
    n_redundant=2,
    n_classes=2,
    random_state=42,
)

# Create synthetic sensitive attribute (e.g., group A vs group B)
# Split roughly 60/40 with label-dependent sampling to create initial bias
sensitive = np.zeros(n_samples, dtype=np.float32)
mask = (X_raw[:, 0] > np.median(X_raw[:, 0])).astype(float)
sensitive = mask.copy()

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X_raw).astype(np.float32)

# Split
X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X, y, sensitive, test_size=0.2, random_state=42, stratify=y,
)

print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")
print(f"Sensitive distribution (train): group 0={sum(s_train==0)}, group 1={sum(s_train==1)}")

# Train baseline model
model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train, y_train)

train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)
print(f"\nBaseline LogisticRegression:")
print(f"  Train accuracy: {train_acc:.4f}")
print(f"  Test accuracy:  {test_acc:.4f}")

In [ ]:
# Cell 3: Compute initial fairness metrics
def compute_fairness_metrics(model, X, y, sensitive):
    """Compute demographic parity gap and equalized odds gap."""
    preds = model.predict(X)
    accuracy = float(np.mean(preds == y))
    
    # Demographic parity: P(y_hat=1 | sensitive=0) vs P(y_hat=1 | sensitive=1)
    unique_groups = np.unique(sensitive)
    group_rates = []
    for g in unique_groups:
        mask = sensitive == g
        if mask.sum() > 0:
            rate = preds[mask].mean()
            group_rates.append(rate)
    dp_gap = float(max(group_rates) - min(group_rates)) if len(group_rates) > 1 else 0.0
    
    # Equalized odds: TPR difference across groups
    eo_gaps = []
    for g in unique_groups:
        mask = sensitive == g
        if mask.sum() > 0:
            tpr = np.mean(preds[mask] == y[mask])
            eo_gaps.append(tpr)
    eo_gap = float(max(eo_gaps) - min(eo_gaps)) if len(eo_gaps) > 1 else 0.0
    
    return {
        'accuracy': accuracy,
        'demographic_parity_gap': dp_gap,
        'equalized_odds_gap': eo_gap,
    }

before_metrics = compute_fairness_metrics(model, X_test, y_test, s_test)

print("Initial Fairness Metrics (before PPO):")
print(f"  Accuracy:               {before_metrics['accuracy']:.4f}")
print(f"  Demographic Parity Gap: {before_metrics['demographic_parity_gap']:.4f}")
print(f"  Equalized Odds Gap:     {before_metrics['equalized_odds_gap']:.4f}")

In [ ]:
# Cell 4: Set up PPO environment and agent
if HAS_TORCH:
    from app.ml.rl.environment import FairnessRLEnvironment
    from app.ml.rl.training_loop import PPOTrainingLoop, TrainingConfig
    
    # Create the fairness RL environment
    env = FairnessRLEnvironment(
        X=X_train,
        y=y_train,
        sensitive_features=s_train,
        model=model,
        max_steps=50,
        target_dp_gap=0.05,
        target_eo_gap=0.05,
        accuracy_floor=0.55,
    )
    
    print(f"Environment created:")
    print(f"  Observation dim: {env.observation_dim}")
    print(f"  Action dim:      {env.action_space.action_dim}")
    print(f"  Max steps:       {env.max_steps}")
    
    # Create PPO training config (small for demo)
    config = TrainingConfig(
        n_episodes=10,             # Small for demo
        max_steps_per_episode=50,
        target_dp_gap=0.05,
        target_eo_gap=0.05,
        accuracy_floor=0.55,
        learning_rate=3e-4,
        gamma=0.99,
        clip_epsilon=0.2,
        early_stopping_patience=5,
        rollout_steps=512,
    )
    
    # Test environment reset/step
    obs = env.reset()
    print(f"\nEnvironment test:")
    print(f"  Observation shape: {obs.shape}")
    print(f"  Observation sample: {obs[:5]}")
    
else:
    print("PyTorch not available. Cannot create PPO environment.")

In [ ]:
# Cell 5: Train PPO agent
training_result = None

if HAS_TORCH:
    print("Starting PPO training...")
    print(f"  Episodes: {config.n_episodes}")
    print(f"  Max steps/episode: {config.max_steps_per_episode}")
    
    t0 = time.time()
    
    loop = PPOTrainingLoop(env=env, config=config)
    training_result = loop.train()
    
    elapsed = time.time() - t0
    
    print(f"\nTraining complete in {elapsed:.2f}s")
    print(f"  Episodes completed:  {training_result.total_episodes}")
    print(f"  Total steps:         {training_result.total_steps}")
    print(f"  Target reached:      {training_result.success}")
    print(f"  Best accuracy:       {training_result.best_accuracy:.4f}")
    print(f"  Best DP gap:         {training_result.best_dp_gap:.4f}")
    print(f"  Best EO gap:         {training_result.best_eo_gap:.4f}")
    print(f"  Best calibration:    {training_result.best_calibration:.4f}")
else:
    print("PyTorch not available. Skipping PPO training.")

In [ ]:
# Cell 6: Compare before/after metrics
print("=" * 60)
print("PPO Bias Mitigation - Before vs After")
print("=" * 60)

if training_result:
    print(f"\n{'Metric':<25} {'Before':>10} {'After':>10} {'Change':>10}")
    print("-" * 55)
    
    acc_change = training_result.best_accuracy - before_metrics['accuracy']
    dp_change = before_metrics['demographic_parity_gap'] - training_result.best_dp_gap
    eo_change = before_metrics['equalized_odds_gap'] - training_result.best_eo_gap
    
    print(f"{'Accuracy':<25} {before_metrics['accuracy']:>10.4f} {training_result.best_accuracy:>10.4f} {acc_change:>+10.4f}")
    print(f"{'Demographic Parity Gap':<25} {before_metrics['demographic_parity_gap']:>10.4f} {training_result.best_dp_gap:>10.4f} {dp_change:>+10.4f}")
    print(f"{'Equalized Odds Gap':<25} {before_metrics['equalized_odds_gap']:>10.4f} {training_result.best_eo_gap:>10.4f} {eo_change:>+10.4f}")
    
    print(f"\nTarget fairness reached: {training_result.success}")
    
    if training_result.best_thresholds:
        print(f"\nLearned thresholds: {training_result.best_thresholds[:5]}")
    if training_result.best_weights:
        print(f"Learned weights (first 5): {training_result.best_weights[:5]}")
else:
    print("\nNo training result available for comparison.")

In [ ]:
# Cell 7: Visualize training curves
if training_result and training_result.metrics_history:
    import matplotlib.pyplot as plt
    
    metrics = training_result.metrics_history
    episodes = [m['episode'] for m in metrics]
    rewards = [m['reward'] for m in metrics]
    accuracies = [m['accuracy'] for m in metrics]
    dp_gaps = [m['dp_gap'] for m in metrics]
    eo_gaps = [m['eo_gap'] for m in metrics]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Reward curve
    axes[0, 0].plot(episodes, rewards, 'b-', linewidth=1.5)
    axes[0, 0].set_title('Episode Reward')
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Cumulative Reward')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy curve
    axes[0, 1].plot(episodes, accuracies, 'g-', linewidth=1.5, label='PPO')
    axes[0, 1].axhline(y=before_metrics['accuracy'], color='r', linestyle='--', 
                        label='Baseline', alpha=0.7)
    axes[0, 1].set_title('Accuracy')
    axes[0, 1].set_xlabel('Episode')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # DP gap curve
    axes[1, 0].plot(episodes, dp_gaps, 'm-', linewidth=1.5, label='PPO')
    axes[1, 0].axhline(y=before_metrics['demographic_parity_gap'], color='r', 
                        linestyle='--', label='Baseline', alpha=0.7)
    axes[1, 0].axhline(y=0.05, color='green', linestyle=':', 
                        label='Target', alpha=0.7)
    axes[1, 0].set_title('Demographic Parity Gap')
    axes[1, 0].set_xlabel('Episode')
    axes[1, 0].set_ylabel('DP Gap')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # EO gap curve
    axes[1, 1].plot(episodes, eo_gaps, 'c-', linewidth=1.5, label='PPO')
    axes[1, 1].axhline(y=before_metrics['equalized_odds_gap'], color='r', 
                        linestyle='--', label='Baseline', alpha=0.7)
    axes[1, 1].axhline(y=0.05, color='green', linestyle=':', 
                        label='Target', alpha=0.7)
    axes[1, 1].set_title('Equalized Odds Gap')
    axes[1, 1].set_xlabel('Episode')
    axes[1, 1].set_ylabel('EO Gap')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle('PPO Training Curves - Bias Mitigation', fontsize=14)
    plt.tight_layout()
    plt.show()
    print("Training curves rendered above.")
else:
    print("No training history available for visualization.")

In [ ]:
# Cell 8: Summary
print("=" * 60)
print("RL Autopilot Training - Summary")
print("=" * 60)

if training_result:
    print(f"""
Data:
  - {n_samples} samples, {n_features} features
  - Sensitive attribute: synthetic binary (group 0 vs group 1)

Baseline model: LogisticRegression
  - Accuracy:               {before_metrics['accuracy']:.4f}
  - Demographic Parity Gap: {before_metrics['demographic_parity_gap']:.4f}
  - Equalized Odds Gap:     {before_metrics['equalized_odds_gap']:.4f}

PPO Training:
  - Episodes: {training_result.total_episodes}
  - Total steps: {training_result.total_steps}
  - Success (target reached): {training_result.success}

After PPO:
  - Accuracy:               {training_result.best_accuracy:.4f}
  - Demographic Parity Gap: {training_result.best_dp_gap:.4f}
  - Equalized Odds Gap:     {training_result.best_eo_gap:.4f}

Key Takeaways:
  1. The PPO agent learns to trade off accuracy for fairness.
  2. In production, use more episodes (50+) for stable convergence.
  3. The Goodhart's Law guard prevents over-optimization of metrics.
  4. Pareto reward modification balances multiple fairness objectives.
""")
else:
    print("\nNo training was performed (PyTorch required).")
    print("Install PyTorch to run PPO training: pip install torch")